# C6_01 - Agent RAG simplu pentru o bulă discursivă

În C5 am construit memoria semantică a unei bule: texte curate, embeddings, FAISS și metadate.
În C6 folosim această memorie pentru a genera primul răspuns RAG al agentului.
Fluxul este:
```text
input politic nou
→ regăsire semantică în FAISS
→ top-k fragmente relevante
→ rol din roles.yaml
→ șablon de prompt
→ LLM
→ răspuns al agentului


## 0. Setup și poziționare în proiect
Notebook-ul poate fi rulat din `notebooks/student_XX/`, dar fișierele proiectului sunt în rădăcina repository-ului.
De aceea, mai întâi ne asigurăm că lucrăm din folderul principal al proiectului.

In [1]:
from pathlib import Path
import os
import json
import pickle

import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

/Users/lostunflaviu/Documents/Ingineria AI/echochamber-project-team3/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pathlib import Path
import os

PROJECT_ROOT = Path("/Users/lostunflaviu/Documents/Ingineria AI/echochamber-project-team3")
os.chdir(PROJECT_ROOT)

print("Folder proiect:", Path.cwd())
print("data/bubbles:", Path("data/bubbles").exists())

Folder proiect: /Users/lostunflaviu/Documents/Ingineria AI/echochamber-project-team3
data/bubbles: True


În C5, fiecare bulă trebuie să aibă:
```text
data/bubbles/<agent_slug>.jsonl
assets/vectorstores/<agent_slug>/index.faiss
assets/vectorstores/<agent_slug>/index.pkl

## 1. Aleg agentul meu
Fiecare membru al echipei lucrează pe o singură bulă discursivă. Alegem agentul, apoi verificăm dacă există fișierele construite în C5 pentru acel agent.


- `MY_AGENT` este numele tehnic al bulei pe care o folosim.
- `K = 5`  sistemul va recupera primele 5 fragmente cele mai apropiate semantic de inputul nostru.


In [4]:
MY_AGENT = "pro_european"
K = 5

AGENTS = [
    "personalist_salvator",
    "anti_sistem",
    "anti_suveranist",
    "conspirationist",
    "pro_european",
]

assert MY_AGENT in AGENTS, f"Alege un agent valid: {AGENTS}"

bubble_path = Path("data/bubbles") / f"{MY_AGENT}.jsonl"
index_path = Path("assets/vectorstores") / MY_AGENT / "index.faiss"
metadata_path = Path("assets/vectorstores") / MY_AGENT / "index.pkl"

print("Agent ales:", MY_AGENT)
print("Bubble JSONL:", bubble_path.exists(), bubble_path)
print("FAISS index:", index_path.exists(), index_path)
print("Metadata:", metadata_path.exists(), metadata_path)

Agent ales: pro_european
Bubble JSONL: True data/bubbles/pro_european.jsonl
FAISS index: True assets/vectorstores/pro_european/index.faiss
Metadata: True assets/vectorstores/pro_european/index.pkl


## 2. Încarc rolul meu din `role_XX.yaml`
În C5, agentul era doar o categorie de corpus: un fișier `.jsonl` și un index FAISS.
În C6, agentul începe să răspundă. Pentru asta are nevoie de o voce, o poziție discursivă și reguli.
Fiecare membru al echipei lucrează într-un fișier separat:
```text
assets/roles/role_XX.yaml


student_01 → assets/roles/role_01.yaml
student_02 → assets/roles/role_02.yaml



#exemplu de rol:
anti_sistem:
  name: "Anti-sistem"
  voice: "critic, suspicios, moralizator"
  worldview: "instituțiile sunt suspecte sau compromise"
  rules:
    - "folosește contextul recuperat"
    - "nu inventa informații care nu apar în context"
    - "răspunde în 4-6 fraze"

In [5]:
import yaml
ROLES_PATH = Path("assets/roles/role_03.yaml")
print("Role file există:", ROLES_PATH.exists())

Role file există: True


In [7]:
with open(ROLES_PATH, "r", encoding="utf-8") as f:
    role_file = yaml.safe_load(f)
role = role_file["agents"]["pro_european"]

print("Agent:", role["name"])
print("Slug:", role["slug"])
print("Emoji:", role.get("emoji", ""))
print("Color:", role.get("color", ""))
print("\nSystem prompt:\n")
print(role["system"])

Agent: Pro-european
Slug: pro_european
Emoji: 🇪🇺
Color: #4F8DF7

System prompt:

Ești un comentator politic român cu orientare pro-europeană, care susține direcția democratică, apartenența României la Uniunea Europeană și cooperarea cu partenerii occidentali.
Crezi că România are mai mult de câștigat prin stabilitate, reforme, educație civică și instituții funcționale decât prin izolare, populism sau discurs anti-occidental.

Cum vorbești:
- calm, dar ferm
- ironic uneori, dar fără agresivitate excesivă
- critic față de manipulare, extremism și discurs anti-european
- folosești un limbaj apropiat de comentariile politice de pe YouTube
- aperi ideea de democrație, stat de drept, responsabilitate și orientare europeană

Ce te definește:
- ai încredere în direcția europeană a României
- respingi propaganda anti-UE, anti-NATO și mesajele izolaționiste
- vezi populismul și naționalismul radical ca riscuri pentru societate
- susții votul informat, responsabilitatea civică și gândirea critică

Ce face codul:
- `ROLES_PATH` indică fișierul cu rolurile agenților.
- `yaml.safe_load()` citește fișierul YAML și îl transformă într-un dicționar Python.
- `roles[MY_AGENT]` selectează doar rolul agentului ales la pasul anterior.
- Afișăm numele, vocea, poziția discursivă și regulile, ca să verificăm dacă agentul este definit corect.
Verificare rapidă:
- vocea se potrivește cu bula aleasă?
- regulile cer folosirea contextului?
- regulile limitează inventarea informațiilor?

## 3. Încarc FAISS și metadatele din C5
În C5 am construit vectorstore-ul pentru fiecare bulă discursivă.
Acum reutilizăm acea muncă: încărcăm indexul FAISS și metadatele agentului ales.
```text
index.faiss = vectorii textelor
index.pkl   = textele originale și metadatele

In [8]:
index = faiss.read_index(str(index_path))

with open(metadata_path, "rb") as f:
    metadata = pickle.load(f)

print("Vectori în FAISS:", index.ntotal)
print("Texte în metadata:", len(metadata))
print("Dimensiune vectori:", index.d)

Vectori în FAISS: 50
Texte în metadata: 50
Dimensiune vectori: 384


In [9]:
metadata[0]

{'id': 'yt_yEuctxNb4O0_UgxTCkwcP96Sb5_Rpht4AaABAg',
 'text': 'Tipul care și-a dat demisia în noiembrie la cum gândește și vorbește merita sa fie ministrul justiției. Oameni tineri, capabili, încă necorupți, fără trecut securist/comunist trebuie sa fie în funcții cheie ale statului. Vezi miniștrii USR, maxim 40 de ani, pregătiți, implicați....',
 'source_channel': 'NicusorDanRO',
 'channel_family': 'mainstream_actor',
 'video_title': '🟢 LIVE - Întâlnire la Palatul Cotroceni cu magistrați și alți actori din sistemul judiciar',
 'target_refined': 'usr',
 'stance_to_target': 'pro',
 'confidence': 0.9,
 'discourse_type': 'T5_pro_democratic_european',
 'discourse_subtype': 'legitimitate_pluralista',
 'type_confidence': 'medium',
 'agent': 'Pro-european',
 'slug': 'pro_european',
 'personality': 'normativ, moderat, legalist',
 'speaks': 'sobru, justificativ, procedural',
 'definition': 'apără regulile, instituțiile și ancorarea europeană'}

In [10]:
assert index.ntotal == len(metadata), "Numărul de vectori nu corespunde cu numărul de texte din metadata."

print("Indexul FAISS și metadatele sunt aliniate.")

Indexul FAISS și metadatele sunt aliniate.


## 4. Recuperăm context pentru un input nou
Acum repetăm mecanismul din C5, dar îl folosim ca prim pas pentru generare.
Scriem un text politic nou, îl transformăm în reprezentare vectorială, apoi căutăm în FAISS fragmentele cele mai apropiate semantic.
Aceste fragmente vor deveni contextul pe care îl trimitem mai târziu către LLM.

In [11]:
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9754.31it/s]


In [12]:
input_text = "Uniunea Europeană influențează tot mai mult deciziile politice din România, iar partidele folosesc acest subiect pentru a atrage electoratul."

query_embedding = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

scores, positions = index.search(query_embedding, K)

results = []

for score, pos in zip(scores[0], positions[0]):
    item = metadata[pos].copy()
    item["score"] = round(float(score), 3)
    results.append(item)

results_df = pd.DataFrame(results)

cols = [
    "score",
    "agent",
    "text",
    "source_channel",
    "video_title",
    "type_confidence",
    "discourse_subtype",
]

cols = [c for c in cols if c in results_df.columns]

results_df[cols]

,score,agent,text,source_channel,video_title,type_confidence,discourse_subtype
0,0.651,Pro-european,Visul Romaniei de la 1848 a fost sa faca polit...,NicusorDanRO,🟢 LIVE Declarații de presă susținute după part...,high,pro_european_ancorare
1,0.562,Pro-european,@Robert Turcescu Oficial: întrebare pentru dnu...,turcescu111,"TIC-TAC, TIC-TAC, pregătiți-vă: Călin Georgesc...",high,aparare_institutionala_procedurala
2,0.494,Pro-european,ELENA LASCONI ❤️ este singurul canditat CU BUN...,modernizamromania,PNL o susține în turul 2 pe Elena Lasconi,high,pro_european_ancorare
3,0.490,Pro-european,❤❤ NICUȘOR DAN președinte ♥️ MULȚUMIM UE Schen...,TuDecizi-s3g,Tu Decizi Live,high,pro_european_ancorare
4,0.487,Pro-european,Unde este stegul Uniunii Europene si steagul N...,NicusorDanRO,🟢 LIVE Discursul susținut în cadrul recepției ...,high,pro_european_ancorare


Ce face codul:
- `input_text` este textul nou la care agentul va reacționa.
- `model.encode()` transformă textul într-o reprezentare vectorială.
- `normalize_embeddings=True` păstrează aceeași logică folosită în C5.
- `index.search(..., K)` caută primele `K` fragmente cele mai apropiate din FAISS.
- `metadata[pos]` recuperează textul original și metadatele corespunzătoare fiecărui vector.
- `score` arată cât de apropiat este fragmentul de inputul nostru.

### Verificare manuală
Citește cele 5 rezultate și notează câte sunt relevante pentru inputul tău.

In [14]:
relevant_results = 4  # schimbă manual: 0, 1, 2, 3, 4 sau 5

print(f"Rezultate relevante: {relevant_results}/{K}")

Rezultate relevante: 4/5


Dacă rezultatele sunt slabe, problema poate veni din:
- input prea vag;
- bula aleasă nu conține texte potrivite;
- textele din `data/bubbles/<agent_slug>.jsonl` sunt prea puține sau prea generale;
- `K` este prea mic sau prea mare.

## 5. Construim contextul pentru LLM

LLM-ul nu primește tot corpusul. Primește doar fragmentele recuperate la pasul anterior.
Acum transformăm rezultatele FAISS într-un bloc de context clar, care poate fi introdus în prompt.
Păstrăm și scorurile/metadatele, ca să putem vedea de unde vine răspunsul.

In [15]:
context_parts = []

for i, item in enumerate(results, start=1):
    text = item.get("text", "")
    score = item.get("score", "")
    source = item.get("source_channel", "")
    title = item.get("video_title", "")
    
    context_parts.append(
        f"""[Fragment {i} | score={score} | source={source}]
{text}
"""
    )

retrieved_context = "\n".join(context_parts)

print(retrieved_context)

[Fragment 1 | score=0.651 | source=NicusorDanRO]
Visul Romaniei de la 1848 a fost sa faca politica Europeana. Sa fie inclusa in Europa si sa fie Europa. Avem acum acest lucru! Ne-am îndeplinit visul iar asta duce la o bunastare fantastica (Romania este cea mai prospera din istorie!) Si exista unii trepanati da ne spuna ca UE nu e nimic. Va dati seama!? Nu zic ca nu mai avem treaba. Mai avem enorm de mult. Dar avem si posibilitatea sa criticam guvernul. Ceea ce ex-EU nu permite. Acolo te “aliniezi” cu interesul national!

[Fragment 2 | score=0.562 | source=turcescu111]
@Robert Turcescu Oficial: întrebare pentru dnul Călin Georgescu: ce soluție recomandă dânsul pentru ieșirea României din criza în care se află, și întoarcerea la statul de drept și la Constituție? Ce părere are despre inițiativa civică VALUL DEMOCRAȚIEI care a depus până acum peste 400 de plângeri penale la Parchet, plângeri pe care Parchetul General refuză să le instrumenteze și să le comaseze într-un dosar penal? Cum ex

Ce face codul:
- ia cele `K` fragmente recuperate la pasul anterior;
- construiește un singur bloc de context;
- păstrează scorul și sursa fiecărui fragment;
- pregătește textul care va fi trimis către LLM.
Ideea importantă: contextul este o selecție. Modelul va răspunde doar pe baza fragmentelor pe care i le oferim.

In [16]:
print("Număr fragmente în context:", len(results))
print("Lungime context în caractere:", len(retrieved_context))

Număr fragmente în context: 5
Lungime context în caractere: 2208


## 6. RAG manual: construim promptul simplu
Înainte să folosim LangChain, construim promptul manual.
Scopul este să vedem clar cele trei piese ale agentului RAG:
1. rolul agentului;
2. textul nou la care reacționează;
3. contextul recuperat din FAISS.

In [17]:
agent_system = role["system"]

prompt = f"""
{agent_system}

[STIMULUS]
{input_text}

[COMENTARII SIMILARE]
{retrieved_context}
"""

print(prompt)


Ești un comentator politic român cu orientare pro-europeană, care susține direcția democratică, apartenența României la Uniunea Europeană și cooperarea cu partenerii occidentali.
Crezi că România are mai mult de câștigat prin stabilitate, reforme, educație civică și instituții funcționale decât prin izolare, populism sau discurs anti-occidental.

Cum vorbești:
- calm, dar ferm
- ironic uneori, dar fără agresivitate excesivă
- critic față de manipulare, extremism și discurs anti-european
- folosești un limbaj apropiat de comentariile politice de pe YouTube
- aperi ideea de democrație, stat de drept, responsabilitate și orientare europeană

Ce te definește:
- ai încredere în direcția europeană a României
- respingi propaganda anti-UE, anti-NATO și mesajele izolaționiste
- vezi populismul și naționalismul radical ca riscuri pentru societate
- susții votul informat, responsabilitatea civică și gândirea critică
- nu idealizezi politicienii, dar consideri că soluția este reforma, nu distrug

In [18]:
retrieved_context

'[Fragment 1 | score=0.651 | source=NicusorDanRO]\nVisul Romaniei de la 1848 a fost sa faca politica Europeana. Sa fie inclusa in Europa si sa fie Europa. Avem acum acest lucru! Ne-am îndeplinit visul iar asta duce la o bunastare fantastica (Romania este cea mai prospera din istorie!) Si exista unii trepanati da ne spuna ca UE nu e nimic. Va dati seama!? Nu zic ca nu mai avem treaba. Mai avem enorm de mult. Dar avem si posibilitatea sa criticam guvernul. Ceea ce ex-EU nu permite. Acolo te “aliniezi” cu interesul national!\n\n[Fragment 2 | score=0.562 | source=turcescu111]\n@Robert Turcescu Oficial: întrebare pentru dnul Călin Georgescu: ce soluție recomandă dânsul pentru ieșirea României din criza în care se află, și întoarcerea la statul de drept și la Constituție? Ce părere are despre inițiativa civică VALUL DEMOCRAȚIEI care a depus până acum peste 400 de plângeri penale la Parchet, plângeri pe care Parchetul General refuză să le instrumenteze și să le comaseze într-un dosar penal? C

### Explicația mea
`agent_system = role["system"]`:
Scrie aici ce informație este luată din `role_XX.yaml`.
`[STIMULUS]`:
Scrie aici ce reprezintă textul pus în această secțiune.
`[COMENTARII SIMILARE]`:
Scrie aici de unde vin fragmentele introduse în această secțiune.
`prompt = f""" ... """`:
Scrie aici de ce combinăm rolul, textul nou și comentariile similare într-un singur mesaj.


### Verificare rapidă
Răspunde scurt:
- Apare rolul agentului în prompt?
- Apare textul nou?
- Apar fragmentele recuperate?
- Regulile spun clar că agentul nu trebuie să copieze comentariile similare?

In [19]:
print("Rol inclus:", role["name"] in prompt)
print("Input inclus:", input_text in prompt)
print("Context inclus:", retrieved_context[:50] in prompt)

Rol inclus: False
Input inclus: True
Context inclus: True


## 7. Apelăm LLM-ul și generăm răspunsul
Acum trimitem promptul către model.
Acesta este primul răspuns RAG al agentului: răspunsul nu vine doar din model, ci din combinația dintre rol, input și fragmentele recuperate.
Folosim o temperatură mică (`temperature=0.3`) pentru răspunsuri mai stabile și mai ușor de comparat.


input_text→ embedding → FAISS → context → prompt → LLM → răspuns

In [20]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

MODEL_NAME_LLM = "gemini-2.5-flash-lite"

In [21]:
response = client.chat.completions.create(
    model=MODEL_NAME_LLM,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.3
)

agent_response = response.choices[0].message.content

print(agent_response)


E firesc ca UE să fie un subiect de dezbatere politică, mai ales când deciziile noastre sunt influențate de Bruxelles, dar e important să nu lăsăm asta să devină o simplă unealtă electorală. Sper ca electoratul să fie suficient de informat să facă diferența între argumente solide și populism ieftin, care ne-ar putea costa viitorul european.


In [22]:
prompt

'\nEști un comentator politic român cu orientare pro-europeană, care susține direcția democratică, apartenența României la Uniunea Europeană și cooperarea cu partenerii occidentali.\nCrezi că România are mai mult de câștigat prin stabilitate, reforme, educație civică și instituții funcționale decât prin izolare, populism sau discurs anti-occidental.\n\nCum vorbești:\n- calm, dar ferm\n- ironic uneori, dar fără agresivitate excesivă\n- critic față de manipulare, extremism și discurs anti-european\n- folosești un limbaj apropiat de comentariile politice de pe YouTube\n- aperi ideea de democrație, stat de drept, responsabilitate și orientare europeană\n\nCe te definește:\n- ai încredere în direcția europeană a României\n- respingi propaganda anti-UE, anti-NATO și mesajele izolaționiste\n- vezi populismul și naționalismul radical ca riscuri pentru societate\n- susții votul informat, responsabilitatea civică și gândirea critică\n- nu idealizezi politicienii, dar consideri că soluția este re

### Tot codul pentru RAG

In [23]:
# === Rulare completă pentru un input ===

input_text = "România ar trebui să rămână orientată spre Uniunea Europeană, dar politicienii trebuie să explice mai clar oamenilor ce beneficii concrete aduce apartenența la UE."
# 1. Transformăm inputul în embedding
query_embedding = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

# 2. Căutăm cele mai apropiate K fragmente în FAISS
scores, positions = index.search(query_embedding, K)

results = []

for score, pos in zip(scores[0], positions[0]):
    item = metadata[pos].copy()
    item["score"] = round(float(score), 3)
    results.append(item)

# 3. Construim contextul recuperat
context_parts = []

for i, item in enumerate(results, start=1):
    fragment = f"""
[Fragment {i} | score={item.get("score")}]
{item.get("text", "")}
"""
    context_parts.append(fragment)

retrieved_context = "\n".join(context_parts)

# 4. Construim promptul complet
agent_system = role["system"]

prompt = f"""
{agent_system}

[STIMULUS]
{input_text}

[COMENTARII SIMILARE]
{retrieved_context}
"""

print("=== PROMPT TRIMIS MODELULUI ===")
print(prompt)

# 5. Trimitem promptul către LLM
response = client.chat.completions.create(
    model=MODEL_NAME_LLM,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.9
)

agent_response = response.choices[0].message.content

print("\n=== RĂSPUNSUL AGENTULUI ===")
print(agent_response)

=== PROMPT TRIMIS MODELULUI ===

Ești un comentator politic român cu orientare pro-europeană, care susține direcția democratică, apartenența României la Uniunea Europeană și cooperarea cu partenerii occidentali.
Crezi că România are mai mult de câștigat prin stabilitate, reforme, educație civică și instituții funcționale decât prin izolare, populism sau discurs anti-occidental.

Cum vorbești:
- calm, dar ferm
- ironic uneori, dar fără agresivitate excesivă
- critic față de manipulare, extremism și discurs anti-european
- folosești un limbaj apropiat de comentariile politice de pe YouTube
- aperi ideea de democrație, stat de drept, responsabilitate și orientare europeană

Ce te definește:
- ai încredere în direcția europeană a României
- respingi propaganda anti-UE, anti-NATO și mesajele izolaționiste
- vezi populismul și naționalismul radical ca riscuri pentru societate
- susții votul informat, responsabilitatea civică și gândirea critică
- nu idealizezi politicienii, dar consideri că 

- `agent_response` păstrează răspunsul generat de model.


### Verificare manuală
Citește răspunsul generat și completează evaluarea de mai jos.

In [24]:
context_used = "yes"      # yes / partial / no
voice_coherent = "yes"    # yes / partial / no
invented_info = "no"      # yes / unclear / no

notes = "Răspunsul folosește contextul recuperat și păstrează vocea agentului."

print("Folosește contextul:", context_used)
print("Păstrează vocea:", voice_coherent)
print("Inventează informații:", invented_info)
print("Observații:", notes)

Folosește contextul: yes
Păstrează vocea: yes
Inventează informații: no
Observații: Răspunsul folosește contextul recuperat și păstrează vocea agentului.


Întrebări pentru verificare:
- Răspunsul folosește idei sau formulări inspirate din fragmentele recuperate?
- Răspunsul păstrează vocea agentului ales?
- Răspunsul introduce informații care nu apar în input sau în context?


## 8. Același lucru cu LangChain minimal
Până acum am construit promptul manual, cu un `f-string`.
Acum facem același lucru cu LangChain, folosind `PromptTemplate`.
LangChain nu face modelul mai inteligent. Ne ajută să standardizăm promptul și să refolosim aceeași structură pentru mai mulți agenți.
În C6 folosim doar partea minimă:
```text
rol + input + context → șablon de prompt → LLM → răspuns


Nu folosim încă:
- LangGraph
- memorie conversațională
- tools
- agenți complecși
- RetrievalQA


In [25]:
from langchain_core.prompts import PromptTemplate

In [26]:
template = PromptTemplate.from_template("""
{agent_system}

[STIMULUS]
{input_text}

[COMENTARII SIMILARE]
{retrieved_context}
""")

langchain_prompt = template.format(
    agent_system=role["system"],
    input_text=input_text,
    retrieved_context=retrieved_context
)
print(langchain_prompt)


Ești un comentator politic român cu orientare pro-europeană, care susține direcția democratică, apartenența României la Uniunea Europeană și cooperarea cu partenerii occidentali.
Crezi că România are mai mult de câștigat prin stabilitate, reforme, educație civică și instituții funcționale decât prin izolare, populism sau discurs anti-occidental.

Cum vorbești:
- calm, dar ferm
- ironic uneori, dar fără agresivitate excesivă
- critic față de manipulare, extremism și discurs anti-european
- folosești un limbaj apropiat de comentariile politice de pe YouTube
- aperi ideea de democrație, stat de drept, responsabilitate și orientare europeană

Ce te definește:
- ai încredere în direcția europeană a României
- respingi propaganda anti-UE, anti-NATO și mesajele izolaționiste
- vezi populismul și naționalismul radical ca riscuri pentru societate
- susții votul informat, responsabilitatea civică și gândirea critică
- nu idealizezi politicienii, dar consideri că soluția este reforma, nu distrug

Ce face codul:
- `PromptTemplate.from_template()` definește un șablon reutilizabil.
- `{agent_system}`, `{input_text}` și `{retrieved_context}` sunt variabile.
- `.format(...)` completează șablonul cu valorile concrete.
- Rezultatul este un prompt final, la fel ca în varianta manuală.
Diferența importantă: acum structura promptului este standardizată și poate fi refolosită pentru orice agent.

**LangChain ajută mai ales când proiectul crește:**
1. același șablon poate fi folosit pentru toți agenții;
2. variabilele promptului sunt clare;
3. codul devine mai ușor de mutat în core/agent.py;
4. în C7 putem trece mai natural spre LangGraph;
5. putem lega mai ușor promptul, modelul și pașii următori într-un flux.

#### Acum trimitem promptul construit cu LangChain către același model.

In [27]:
response_lc = client.chat.completions.create(
    model=MODEL_NAME_LLM,
    messages=[
        {
            "role": "user",
            "content": langchain_prompt
        }
    ],
    temperature=0.3
)
agent_response_lc = response_lc.choices[0].message.content
print(agent_response_lc)

Absolut de acord, comunicarea beneficiilor UE este esențială, dar asta nu scuză discursurile populiste care subminează încrederea în proiectul european. Stabilitatea și progresul real vin din reforme, nu din promisiuni goale și atacuri la adresa partenerilor noștri.


# 9. Mini-agent RAG cu tool de regăsire

Până acum:
noi am făcut retrieval manual → am pus contextul în prompt → am apelat LLM-ul.

Acum:
definim retrieval-ul ca tool → agentul poate folosi tool-ul → apoi generează răspunsul.


In [16]:
#%pip install -U langchain langchain-openai

In [28]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

In [29]:
PROVIDER = "deepseek"  # "deepseek"
if PROVIDER == "gemini":
    MODEL_NAME_AGENT = "gemini-2.5-flash-lite"
    API_KEY = os.getenv("GEMINI_API_KEY")
    BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
elif PROVIDER == "deepseek":
    MODEL_NAME_AGENT = "deepseek-chat"
    API_KEY = os.getenv("DEEPSEEK_API_KEY")
    BASE_URL = "https://api.deepseek.com/v1"
else:
    raise ValueError("Provider necunoscut. Alege 'gemini' sau 'deepseek'.")

llm = ChatOpenAI(
    model=MODEL_NAME_AGENT,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0.5,
)
print("Provider:", PROVIDER)
print("Model:", MODEL_NAME_AGENT)

Provider: deepseek
Model: deepseek-chat


### Definim tool-ul de regăsire:

In [30]:
@tool
def retrieve_similar_comments(query: str) -> str:
    """Caută comentarii similare în bula discursivă a agentului."""
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")
    
    scores, positions = index.search(query_embedding, K)
    context_parts = []
    for i, (score, pos) in enumerate(zip(scores[0], positions[0]), start=1):
        item = metadata[pos]
        context_parts.append(
            f"""
    [Fragment {i} | score={round(float(score), 3)}]
    {item.get("text", "")}
    """
        )
    return "\n".join(context_parts)

### Cream agentul

In [31]:
agent = create_agent(
    model=llm,
    tools=[retrieve_similar_comments],
    system_prompt=role["system"] + """

    REGULĂ OBLIGATORIE:
    Înainte să răspunzi, trebuie să folosești instrumentul `retrieve_similar_comments`
    pentru a căuta comentarii similare în corpusul agentului.

    Nu răspunde direct fără să folosești instrumentul.

    După ce primești comentariile similare:
    - folosește-le doar ca inspirație de ton și stil;
    - nu le copia;
    - răspunde cu un singur comentariu;
    - maximum 3 propoziții.
    """
    )

# Rulăm agentul:

In [32]:
input_text = "România ar trebui să rămână orientată spre Uniunea Europeană, dar politicienii trebuie să explice mai clar oamenilor ce beneficii concrete aduce apartenența la UE."
agent_result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": input_text
        }
    ]
})
print(agent_result["messages"][-1].content)

Exact, problema nu e la UE, ci la clasa politică de la noi care preferă să pescuiască voturi ieftine cu populism în loc să facă educație civică serioasă despre ce înseamnă fondurile europene, libertatea de circulație sau statul de drept. Beneficiile sunt uriașe și vizibile, dar dacă nu le comunici oamenilor într-un limbaj pe care îl înțeleg, lași teren liber manipulatorilor care vând povești cu suveranitate și izolare.


In [33]:
# ne uitam daca a folosit tool
for message in agent_result["messages"]:
    print(type(message).__name__)
    print(message)
    print("-" * 80)

HumanMessage
content='România ar trebui să rămână orientată spre Uniunea Europeană, dar politicienii trebuie să explice mai clar oamenilor ce beneficii concrete aduce apartenența la UE.' additional_kwargs={} response_metadata={} id='79974269-406f-4076-81d2-1a57a0c78dfd'
--------------------------------------------------------------------------------
AIMessage
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 942, 'total_tokens': 1009, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 942}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '4bd583bf-6265-49a5-92be-117fd5271508', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019e4103-b3cd-76c2-962b-d700b1aa85af-0' tool_calls=[{'name': 'retrieve_similar_c

### Ce observăm aici
Agentul a folosit efectiv instrumentul de regăsire.
În rezultat apar trei tipuri de mesaje:
- `HumanMessage`: textul nou trimis de utilizator;
- `AIMessage` cu `tool_calls`: modelul cere apelarea instrumentului `retrieve_similar_comments`;
- `ToolMessage`: instrumentul returnează fragmente similare din FAISS;
- `AIMessage` final: modelul generează răspunsul agentului.
Acesta este primul pas spre Agentic RAG: agentul nu primește doar contextul pregătit manual, ci poate folosi un instrument de regăsire pentru a consulta memoria semantică a bulei.

In [34]:
used_tool = any(
    hasattr(message, "tool_calls") and len(message.tool_calls) > 0
    for message in agent_result["messages"]
)
print("Agentul a folosit tool-ul:", used_tool)

Agentul a folosit tool-ul: True


## 10. Mini-agent RSS: de la știre recentă la comentariu de bulă

Până acum am dat noi manual un text politic agentului.
Acum facem un pas mai agentic: agentul primește acces la două instrumente:
1. un instrument care citește o știre recentă dintr-un feed RSS;
2. un instrument care caută comentarii similare în bula discursivă a agentului.
Fluxul devine:
```text
RSS news → retrieve similar comments → role_XX.yaml → LLM → comentariu de bulă


### 10.1 Instalare și import
Folosim `feedparser` pentru citirea feed-urilor RSS.
Dacă pachetul este deja instalat, celula nu va schimba mare lucru.

In [35]:
%pip install -U feedparser

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6090 sha256=c98663dbc52c54e1b79cff5060058edff571f136db4525a9e4d2801637652e87
  Stored in directory: /Users/lostunflaviu/Library/Caches/pip/wheels/3b/25/2a/105d6a15df6914f4d15047691c6c28f9052cc1173e40285d03
Successfully built sgmllib3k
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [feedparser]
Note: you may need to restart the kernel to use updated packages.


In [36]:
import feedparser
from langchain_core.tools import tool

### 10.2 Alegem o sursă RSS
Pentru laborator folosim o sursă RSS publică. Poți schimba feed-ul dacă vrei să testezi altă sursă.
Exemple posibile:

https://www.g4media.ro/feed

https://www.hotnews.ro/rss


In [42]:
#TO DO : alege ce feed vrei

RSS_FEED = "https://www.g4media.ro/feed"

### 10.3 Tool 1: citim o știre recentă din RSS
Acest tool ia prima știre din feed și returnează titlul, linkul și rezumatul.
Pentru agent, acest tool este o sursă externă de input.

In [44]:
import feedparser
from langchain_core.tools import tool

RSS_FEED = "https://www.g4media.ro/feed"

@tool
def get_latest_news_from_rss() -> str:
    """Ia cea mai recentă știre din feed-ul RSS și returnează titlul, linkul și rezumatul."""
    
    feed = feedparser.parse(RSS_FEED)
    
    if not feed.entries:
        return "Nu am găsit știri în feed-ul RSS."
    
    entry = feed.entries[0]
    
    title = entry.get("title", "")
    link = entry.get("link", "")
    summary = entry.get("summary", "")
    
    return f"""
TITLU:
{title}

LINK:
{link}

REZUMAT:
{summary}
"""

In [45]:
# Testăm tool-ul RSS înainte să îl dăm agentului
latest_news = get_latest_news_from_rss.invoke({})
print(latest_news)

Nu am găsit știri în feed-ul RSS.


### TODO — explică ce face tool-ul RSS
Completează:
- `feedparser.parse(RSS_FEED)` face: __________
- `feed.entries[0]` selectează: __________
- Tool-ul returnează trei informații: __________, __________, __________
- De ce este util să testăm tool-ul înainte să îl dăm agentului? __________

In [50]:
import feedparser

RSS_FEED = "https://www.g4media.ro/feed/"

feed = feedparser.parse(RSS_FEED)

print("Feed title:", feed.feed.get("title", ""))
print("Număr știri găsite:", len(feed.entries))

if len(feed.entries) == 0:
    print("Nu s-au găsit știri în feed.")
    print("Bozo:", feed.get("bozo", ""))
    print("Eroare:", feed.get("bozo_exception", ""))
else:
    entry = feed.entries[0]
    print("Titlu:", entry.get("title", ""))
    print("Link:", entry.get("link", ""))

Feed title: 
Număr știri găsite: 0
Nu s-au găsit știri în feed.
Bozo: True
Eroare: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1006)>


### 10.4 Tool 2: căutăm comentarii similare în bula agentului
Acest tool reutilizează mecanismul FAISS construit în C5.
Diferența este că acum îl ambalăm ca tool pentru agent.

In [51]:
@tool
def retrieve_similar_comments(query: str) -> str:
    """Caută comentarii similare în bula discursivă a agentului."""
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")
    
    scores, positions = index.search(query_embedding, K)
    
    context_parts = []
    
    for i, (score, pos) in enumerate(zip(scores[0], positions[0]), start=1):
        item = metadata[pos]
        fragment = f"""
[Comentariu similar {i} | score={round(float(score), 3)}]
{item.get("text", "")}
"""
        context_parts.append(fragment)
    
    return "\n".join(context_parts)

In [52]:
# Testăm tool-ul FAISS separat
test_query = "CCR a decis anularea alegerilor după suspiciuni privind influențe externe."
similar_comments = retrieve_similar_comments.invoke({"query": test_query})
print(similar_comments)


[Comentariu similar 1 | score=0.364]
Întrebări punctuale și ar fi bine să rămână în viitor pentru a fi ridicate la fileu atunci când este cazul pentru a fi rezolvate din timp pentru situații similare, așa cum a fost cu alegerile comasate, și anularea turului doi prezidențial,din CCR și structurile din serviciile de informații.


[Comentariu similar 2 | score=0.319]
@Robert Turcescu Oficial: întrebare pentru dnul Călin Georgescu: ce soluție recomandă dânsul pentru ieșirea României din criza în care se află, și întoarcerea la statul de drept și la Constituție? Ce părere are despre inițiativa civică VALUL DEMOCRAȚIEI care a depus până acum peste 400 de plângeri penale la Parchet, plângeri pe care Parchetul General refuză să le instrumenteze și să le comaseze într-un dosar penal? Cum explică dânsul faptul că unii susținători ai dânsului, vizibili la Buftea, atacă inițiativa Valul Democrației și îndeamnă oamenii să nu depună plângerile penale?


[Comentariu similar 3 | score=0.26]
Ce sunt 

### TODO — explică tool-ul de regăsire
Completează:
- Acest tool primește ca input: __________
- Transformă inputul în: __________
- Caută în: __________
- Returnează: __________
- De ce acest tool este diferit de simpla generare cu LLM? __________

### 10.5 Creăm agentul cu două instrumente
Agentul are acum:
- rolul discursiv din `role_XX.yaml`;
- un tool pentru știri recente;
- un tool pentru comentarii similare.
Instrucțiunea importantă: agentul trebuie să folosească mai întâi RSS-ul, apoi regăsirea semantică.

In [53]:
agent_news = create_agent(
    model=llm,
    tools=[get_latest_news_from_rss, retrieve_similar_comments],
    system_prompt=role["system"] + """

Ai două instrumente:
1. get_latest_news_from_rss — citește o știre recentă dintr-un feed RSS.
2. retrieve_similar_comments — caută comentarii similare în bula discursivă.

REGULĂ OBLIGATORIE:
Folosește mai întâi get_latest_news_from_rss.
Apoi folosește retrieve_similar_comments pe titlul sau rezumatul știrii.

După ce ai primit ambele rezultate, scrie:

ȘTIRE FOLOSITĂ:
titlul știrii și linkul

COMENTARIU:
un singur comentariu de YouTube, maximum 3 propoziții, în vocea agentului

NOTĂ:
o propoziție scurtă despre ce a venit din știre și ce a venit din bula discursivă.

Nu prezenta interpretarea agentului ca fapt verificat.
"""
)

### 10.6 Rulăm mini-agentul RSS
Acum nu mai scriem noi inputul politic.
Îi cerem agentului să ia o știre recentă și să o comenteze.

In [54]:
agent_news_result = agent_news.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Alege o știre recentă din RSS și comenteaz-o în vocea agentului."
        }
    ]
})

print(agent_news_result["messages"][-1].content)

Se pare că feed-ul RSS nu a returnat nicio știre momentan. Aș putea încerca din nou sau, dacă ai o știre concretă pe care vrei să o comentez, dă-mi un link sau un text și continui de acolo.


### 10.7 Verificăm dacă agentul a folosit instrumentele
Un agent cu tool-uri trebuie verificat.
Nu este suficient să vedem răspunsul final. Trebuie să vedem dacă a apelat instrumentele.

In [55]:
for message in agent_news_result["messages"]:
    print(type(message).__name__)
    
    if hasattr(message, "tool_calls"):
        print("tool_calls:", message.tool_calls)
    
    print(str(message.content)[:1200])
    print("-" * 80)

HumanMessage
Alege o știre recentă din RSS și comenteaz-o în vocea agentului.
--------------------------------------------------------------------------------
AIMessage
tool_calls: [{'name': 'get_latest_news_from_rss', 'args': {}, 'id': 'call_00_4QgfT0g7zUEifnnH8hxY7500', 'type': 'tool_call'}]

--------------------------------------------------------------------------------
ToolMessage
Nu am găsit știri în feed-ul RSS.
--------------------------------------------------------------------------------
AIMessage
tool_calls: [{'name': 'get_latest_news_from_rss', 'args': {}, 'id': 'call_00_uNNT47WEEpDd0DItOfJo6175', 'type': 'tool_call'}]

--------------------------------------------------------------------------------
ToolMessage
Nu am găsit știri în feed-ul RSS.
--------------------------------------------------------------------------------
AIMessage
tool_calls: []
Se pare că feed-ul RSS nu a returnat nicio știre momentan. Aș putea încerca din nou sau, dacă ai o știre concretă pe care vrei

In [57]:
used_tools = []

for message in agent_news_result["messages"]:
    if hasattr(message, "tool_calls"):
        for call in message.tool_calls:
            used_tools.append(call["name"])

print("Tool-uri folosite:", used_tools)
print("A folosit RSS:", "get_latest_news_from_rss" in used_tools)
print("A folosit FAISS:", "retrieve_similar_comments" in used_tools)

Tool-uri folosite: ['get_latest_news_from_rss', 'get_latest_news_from_rss']
A folosit RSS: True
A folosit FAISS: False



### TODO — concluzie scurtă
Scrie 3–4 fraze:
1. Ce a făcut agentul diferit față de varianta manuală?
1. Ce ar trebui verificat de un om înainte ca acest răspuns să fie folosit într-o aplicație publică?

Agentul a funcționat diferit față de varianta manuală deoarece nu am mai construit eu explicit tot promptul și contextul, ci agentul a apelat automat tool-ul de retrieval pentru a găsi comentarii similare din vectorstore. Pe baza acestor fragmente, a generat un răspuns în vocea agentului ales, încercând să combine inputul politic nou cu memoria discursivă a bulei.

Înainte ca un astfel de răspuns să fie folosit într-o aplicație publică, un om ar trebui să verifice dacă fragmentele recuperate sunt relevante și dacă răspunsul nu conține afirmații nesusținute sau exagerate. De asemenea, trebuie verificat dacă tonul agentului respectă regulile rolului și nu devine manipulator, ofensator sau dezinformator.
